# Stage 02: Exploratory Data Analysis & Consumption Profiling

## Overview
This notebook analyzes electricity consumption behaviors across meters:
- Visualizing temporal profiles across day-of-week and time-of-day
- Comparing residential/small commercial (`MT_001`) with heavy power consumer (`MT_196`)
- Identifying cyclical demand structures (diurnal peaks and nightly troughs)
- Exporting visualization artifacts to `outputs/figures/`

In [ ]:
import csv
from datetime import datetime, timedelta
from pathlib import Path
import matplotlib.pyplot as plt

# Paths
root = Path('..') if Path('..').resolve().name == 'ML-MINI-PROJECT' else Path('.')
clean_file = root / 'data' / 'processed' / 'LD2011_2014_clean.csv'
fig_dir = root / 'outputs' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

print(f"Clean data exists: {clean_file.exists()}")

## 1. First Week Consumption Profile: Client MT_001
Let's extract and plot the first active week for client `MT_001` to observe short-term dynamics and 15-minute resolution patterns.

In [ ]:
times = []
readings = []
first_time = None
client = "MT_001"

with clean_file.open(encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        t = datetime.strptime(row["timestamp"], "%Y-%m-%d %H:%M:%S")
        val = float(row[client])
        # Find first non-zero reading or start of time series
        if first_time is None:
            first_time = t
        if t >= first_time + timedelta(days=7):
            break
        times.append(t)
        readings.append(val)

plt.figure(figsize=(10, 4))
plt.plot(times, readings, color='#1f77b4', linewidth=1.2)
plt.title(f"{client} Electricity Use During the First Week", fontsize=12, fontweight='bold')
plt.xlabel("Timestamp")
plt.ylabel("Consumption (kW)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

out_chart1 = fig_dir / "first_week_MT_001.png"
plt.savefig(out_chart1, dpi=150)
plt.show()
print(f"Saved figure to {out_chart1}")

## 2. Average Daily Profile (24-Hour Cycle): Client MT_196
Client `MT_196` represents a high-load electricity consumer.
Calculating average consumption per hour reveals the expected diurnal pattern:
- Low overnight consumption (trough)
- Rapid morning ramp-up around 08:00
- Sustained daytime plateau
- Evening decrease

In [ ]:
client = "MT_196"
hour_totals = [0.0] * 24
hour_counts = [0] * 24

with clean_file.open(encoding='utf-8', newline='') as f:
    reader = csv.reader(f)
    header = next(reader)
    idx = header.index(client)
    for row in reader:
        h = int(row[0][11:13])
        hour_totals[h] += float(row[idx])
        hour_counts[h] += 1

hour_averages = [hour_totals[h] / hour_counts[h] for h in range(24)]

plt.figure(figsize=(9, 4.5))
plt.plot(range(24), hour_averages, marker='o', color='#2ca02c', linewidth=2, markersize=5)
plt.title(f"Average Daily Electricity Profile: {client}", fontsize=12, fontweight='bold')
plt.xlabel("Hour of Day (0-23)")
plt.ylabel("Average Consumption (kW)")
plt.xticks(range(0, 24, 2))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

out_chart2 = fig_dir / f"average_daily_profile_{client}.png"
plt.savefig(out_chart2, dpi=150)
plt.show()
print(f"Saved figure to {out_chart2}")

## 3. Distribution & Outlier Characteristics
Summary statistics highlight substantial variance and positive skewness:
- Standard deviation is sizable due to diurnal swings.
- Occasional extreme spikes reach multiples of the median baseline.
- These strong periodicities indicate that raw values alone cannot determine anomalies; time context and dynamic baselines are crucial.